In [0]:
from pyspark.sql.functions import first, rand, count
dates = spark.sql("SELECT explode(sequence(DATE'2024-01-01', DATE'2024-03-24', INTERVAL 1 DAY)) as  calendar_date") #explode rozpakowuje liste czyli tworzy rekordy pomiedzy zakresami
c_id = spark.sql("SELECT explode(sequence(1,400, 1)) as  client_id")
types = spark.sql("""SELECT concat("col_", colName) as col_name from (SELECT explode(sequence(1,40, 1)) as  colName)""")
 #partycja danych wazne dla podzialu przy joinach i duzych operacjach
dates = dates.repartition(99)
c_id = c_id.repartition(11)
types = types.repartition(1)
 # polaczenie unikalnycg client_id, calendar_date, col_name wszystko do wszystkiego od wszystkiego
df_cartesian = c_id.crossJoin(dates.select("calendar_date")).crossJoin(types.select("col_name")).select("client_id","calendar_date","col_name")

# ilosc client_id na dane dni zawsze 4000 w tym przypadku 200 klientów * 20 kolumn
df_cartesian2 = df_cartesian.groupBy("calendar_date").agg(count("client_id"))
 
# display(df_cartesian2.limit(1000))
 
df_cartesian = df_cartesian.withColumn("val", (rand()*10).cast("int"))
df_cartesian_v2 = df_cartesian.withColumn("val", (rand()*10).cast("int"))
df_grp = df_cartesian.groupBy("client_id","calendar_date").pivot("col_name").agg((first("val").alias("val")))

display(df_grp.limit(100))


client_id,calendar_date,col_1,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_2,col_20,col_21,col_22,col_23,col_24,col_25,col_26,col_27,col_28,col_29,col_3,col_30,col_31,col_32,col_33,col_34,col_35,col_36,col_37,col_38,col_39,col_4,col_40,col_5,col_6,col_7,col_8,col_9
161,2024-03-19,7,8,0,2,6,9,6,3,5,0,2,4,3,8,2,6,5,3,3,0,4,7,0,6,8,0,6,1,9,1,4,0,1,6,1,8,2,3,0,0
161,2024-03-07,3,6,3,5,1,1,8,1,7,0,0,9,5,3,6,1,1,8,5,3,2,6,5,3,5,6,6,6,1,9,4,8,6,5,4,8,7,6,1,6
161,2024-01-17,6,0,2,8,9,4,1,1,8,1,1,7,9,6,6,7,4,3,4,0,5,7,8,4,2,5,4,6,5,9,7,1,8,2,2,3,1,2,7,4
161,2024-01-31,1,0,0,8,6,8,7,0,8,0,1,6,6,3,4,2,6,1,9,3,0,0,3,2,3,6,3,5,5,9,4,0,6,4,3,4,5,4,1,0
161,2024-02-26,8,2,3,1,5,8,0,3,7,2,7,9,2,6,1,9,6,1,2,1,2,9,1,0,8,1,9,3,8,6,8,0,8,9,2,0,5,0,0,2
161,2024-03-21,7,3,6,4,2,1,7,5,0,9,6,9,8,8,4,4,0,5,3,6,0,4,9,8,6,7,0,1,7,3,7,1,9,2,8,0,9,3,2,1
161,2024-01-29,8,7,9,8,9,5,0,8,6,7,9,2,4,3,7,9,7,2,7,5,3,3,9,5,4,2,7,8,6,5,1,4,1,8,1,9,2,1,7,5
161,2024-03-24,4,8,2,8,6,2,3,5,1,2,8,7,0,9,3,3,3,1,7,1,0,3,8,1,0,6,2,1,9,1,4,9,3,5,1,7,3,6,9,0
161,2024-02-24,3,4,5,7,3,0,2,1,7,6,4,2,2,1,0,6,5,1,7,7,9,7,9,4,4,6,4,0,8,3,9,2,4,2,4,9,3,8,3,3
161,2024-02-29,5,2,8,0,6,4,6,8,6,8,7,5,9,3,2,2,0,5,8,6,2,6,9,6,8,6,7,6,7,8,1,2,9,0,7,0,4,7,5,4


In [0]:
df_cartesian.limit(10).show()
df_join = df_cartesian.join(df_cartesian_v2, (df_cartesian.calendar_date == df_cartesian_v2.calendar_date) & 
                  (df_cartesian.val == df_cartesian_v2.val) & 
                  (df_cartesian.client_id == df_cartesian_v2.client_id),"left")
display(df_join.limit(100))

+---------+-------------+--------+---+
|client_id|calendar_date|col_name|val|
+---------+-------------+--------+---+
|      161|   2024-03-19|   col_1|  6|
|      161|   2024-03-19|   col_2|  2|
|      161|   2024-03-19|   col_3|  4|
|      161|   2024-03-19|   col_4|  4|
|      161|   2024-03-19|   col_5|  4|
|      161|   2024-03-19|   col_6|  9|
|      161|   2024-03-19|   col_7|  2|
|      161|   2024-03-19|   col_8|  7|
|      161|   2024-03-19|   col_9|  7|
|      161|   2024-03-19|  col_10|  4|
+---------+-------------+--------+---+



client_id,calendar_date,col_name,val,client_id,calendar_date,col_name,val
161,2024-01-01,col_1,4,null,null,null,null
108,2024-01-01,col_1,6,108,2024-01-01,col_17,6
161,2024-01-02,col_1,7,161,2024-01-02,col_1,7
161,2024-01-02,col_1,7,161,2024-01-02,col_8,7
161,2024-01-02,col_1,7,161,2024-01-02,col_9,7
161,2024-01-02,col_1,7,161,2024-01-02,col_32,7
161,2024-01-02,col_1,7,161,2024-01-02,col_39,7
161,2024-01-03,col_1,2,161,2024-01-03,col_5,2
161,2024-01-03,col_1,2,161,2024-01-03,col_6,2
161,2024-01-03,col_1,2,161,2024-01-03,col_28,2


In [0]:
df_join = df_cartesian.join(df_cartesian_v2, (df_cartesian.calendar_date == df_cartesian_v2.calendar_date) & 
                  (df_cartesian.val == df_cartesian_v2.val) & 
                  (df_cartesian.client_id == df_cartesian_v2.client_id),"inner")
display(df_join.limit(100))

client_id,calendar_date,col_name,val,client_id,calendar_date,col_name,val
1,2024-01-01,col_14,0,1,2024-01-01,col_10,0
1,2024-01-01,col_14,0,1,2024-01-01,col_13,0
1,2024-01-01,col_14,0,1,2024-01-01,col_25,0
1,2024-01-01,col_14,0,1,2024-01-01,col_26,0
1,2024-01-01,col_14,0,1,2024-01-01,col_39,0
1,2024-01-01,col_16,0,1,2024-01-01,col_10,0
1,2024-01-01,col_16,0,1,2024-01-01,col_13,0
1,2024-01-01,col_16,0,1,2024-01-01,col_25,0
1,2024-01-01,col_16,0,1,2024-01-01,col_26,0
1,2024-01-01,col_16,0,1,2024-01-01,col_39,0


ROZNICE MIEDZY LEFT I INNER:
- roznica w sortowaniu wynikow w left mamy posortowana lewa tabele od wsyzskich rekordow malejaco, roznica w col_\d 
- tabela left zawiera wartosci null dla rekordow ktore nie maja dopasowan w prawej tabeli

In [0]:
#df_cartesian_v2 = df_cartesian_v2.withColumnRenamed("calendar_date","c_date")
df_join = df_cartesian.join(df_cartesian_v2, "calendar_date")
display(df_join.limit(100))

calendar_date,client_id,col_name,val,client_id,col_name,val
2024-01-01,161,col_1,4,161,col_1,2
2024-01-01,161,col_1,4,108,col_1,4
2024-01-01,161,col_1,4,276,col_1,4
2024-01-01,161,col_1,4,302,col_1,4
2024-01-01,161,col_1,4,355,col_1,3
2024-01-01,161,col_1,4,65,col_1,3
2024-01-01,161,col_1,4,370,col_1,8
2024-01-01,161,col_1,4,238,col_1,4
2024-01-01,161,col_1,4,342,col_1,9
2024-01-01,161,col_1,4,327,col_1,4


In [0]:
df_join = df_cartesian.join(df_cartesian_v2, df_cartesian.calendar_date == df_cartesian_v2.calendar_date,"inner").drop(df_join.calendar_date)
display(df_join.limit(100))

client_id,col_name,val,client_id,calendar_date,col_name,val
161,col_1,4,161,2024-01-01,col_1,2
161,col_1,4,108,2024-01-01,col_1,4
161,col_1,4,276,2024-01-01,col_1,4
161,col_1,4,302,2024-01-01,col_1,4
161,col_1,4,355,2024-01-01,col_1,3
161,col_1,4,65,2024-01-01,col_1,3
161,col_1,4,370,2024-01-01,col_1,8
161,col_1,4,238,2024-01-01,col_1,4
161,col_1,4,342,2024-01-01,col_1,9
161,col_1,4,327,2024-01-01,col_1,4
